In [1]:
# Resstock model download, and ami optimization 

"""
Created on April 2 11:00:00 2026

@author: Tanushree Charan

"""

'\nCreated on April 2 11:00:00 2026\n\n@author: Tanushree Charan\n\n'

In [2]:
pip install awscli


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pyarrow fastparquet


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
!aws --version

aws-cli/1.42.0 Python/3.13.2 Darwin/25.4.0 botocore/1.40.0


In [5]:
pip install pandas


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import subprocess
import zipfile
import os
import shutil
import json
import ast
import pandas as pd
from pathlib import Path

In [7]:
# SCRIPT TO READ AND SAVE ALL BUILDING IDS FROM THE NY METADATA FILE 2025(FILTERED FOR DRYDEN RELEVANT COUNTIES)
# Read list of all building IDs

# path to metadata parquet
current_path = Path(os.getcwd())
data_path = current_path / '..'/ 'data' 
buildstock_metadata_path = '/Users/tcharan/Library/CloudStorage/OneDrive-NREL/NRELWorkPostJune2024/Projects/Grid Edge/FY25_Grid Edge/2025 ResStock/NY_upgrade0.xlsx'
# Read the "select_counties" sheet
df = pd.read_excel(buildstock_metadata_path, sheet_name="select_counties")

# Number of rows in the DataFrame
num_rows = len(df)
print("Number of rows:", num_rows)

building_ids_metadata = df["bldg_id"]

# Save building IDs to CSV
output_path = data_path / "base_files" / "building_ids_metadata_2025.csv"
building_ids_metadata.to_csv(output_path, index=False)

print("bldg_id count:", len(building_ids_metadata))
print(building_ids_metadata)
print('done')


Number of rows: 1516
bldg_id count: 1516
0          402
1          486
2          649
3         1387
4         1572
         ...  
1511    549227
1512    549378
1513    549749
1514    549921
1515    549976
Name: bldg_id, Length: 1516, dtype: int64
done


In [8]:
# SCRIPT TO READ AND STORE ALL BUILDING IDS FOR THE NY METADATA FILE (FILTERED FOR DRYDEN RELEVANT COUNTIES)
# SCRIPT STORES LIST OF BUILDING IDS
# read metadata IDs
current_path = Path(os.getcwd())
data_path = current_path / '..'/ 'data'
path = data_path / "base_files" / "building_ids_metadata_2025.csv"
df_bid = pd.read_csv(path)
building_ids = df_bid["bldg_id"]
print("bldg_id count:", len(building_ids))

bldg_id count: 1516


In [9]:
# --- Faster ResStock downloader: parallel + multipart S3 ---
import os, json, zipfile, sys, subprocess, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# ========================
# CONFIG
# ========================
BUILDING_IDS = [140567, 253852, 379341]   # <--- edit
S3_BASE = ("s3://oedi-data-lake/nrel-pds-building-stock/"
           "end-use-load-profiles-for-us-building-stock/2024/"
           "resstock_tmy3_release_2/model_and_schedule_files/"
           "building_energy_models/upgrade=0/")

BASE_FILES = Path("../data/base_files")
MEASURES_OG = BASE_FILES / "measures"
WORKFLOW_OG = BASE_FILES / "workflow_resstock.osw"
WEATHER_FILE = "Ithaca_2024.epw"
WEATHER_OG = BASE_FILES / "weather" / WEATHER_FILE

EXTRACT_DIR = Path("../data/resstock_models")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 6          # try 4–8 depending on your bandwidth/CPU
CHUNK_MB = 8             # multipart chunk size for S3 (MB)

# ========================
# S3 DOWNLOAD IMPL (boto3 if available -> fast; else awscli module)
# ========================
_use_boto3 = False
try:
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config
    from boto3.s3.transfer import TransferConfig, S3Transfer
    _use_boto3 = True
    _s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))
    _transfer = S3Transfer(
        _s3,
        TransferConfig(
            multipart_threshold=CHUNK_MB*1024*1024,
            multipart_chunksize=CHUNK_MB*1024*1024,
            max_concurrency=10,
            use_threads=True
        )
    )
except Exception:
    pass

AWS_PREFIX = [sys.executable, "-m", "awscli"]  # fallback

def s3_url_parts(s3_url: str):
    # "s3://bucket/key..." -> ("bucket","key")
    assert s3_url.startswith("s3://")
    rest = s3_url[5:]
    bucket, key = rest.split("/", 1)
    return bucket, key

def download_s3(s3_url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if _use_boto3:
        bucket, key = s3_url_parts(s3_url)
        _transfer.download_file(bucket, key, str(dest))
    else:
        cmd = AWS_PREFIX + ["s3", "cp", s3_url, str(dest), "--no-sign-request", "--only-show-errors"]
        subprocess.run(cmd, check=True)

# ========================
# UTILITIES
# ========================
def ensure_base_files():
    for p in [MEASURES_OG, WORKFLOW_OG, WEATHER_OG]:
        if not p.exists():
            raise FileNotFoundError(f"Missing base file/dir: {p}")

def unzip_to_dir(zip_path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)

def copy_weather_and_measures(local_extract_path: Path):
    (local_extract_path / "weather").mkdir(exist_ok=True)
    shutil.copy(WEATHER_OG, local_extract_path / "weather")

    measures_dst = local_extract_path / "measures"
    if measures_dst.exists():
        shutil.rmtree(measures_dst)
    shutil.copytree(MEASURES_OG, measures_dst)

def patch_osw(local_extract_path: Path):
    osw_dst = local_extract_path / "workflow_resstock.osw"
    shutil.copy(WORKFLOW_OG, osw_dst)
    with open(osw_dst, "r") as f:
        data = json.load(f)
    data["seed_file"] = "in.osm"
    data["weather_file"] = WEATHER_FILE
    with open(osw_dst, "w") as f:
        json.dump(data, f, indent=2)

def building_done(local_extract_path: Path) -> bool:
    # crude skip: consider done if models dir exists and has files
    models = local_extract_path / "models"
    return models.exists() and any(models.iterdir())

def process_building(bid: int):
    padded = f"{bid:07d}"
    filename = f"bldg{padded}-up00.zip"
    s3_url = S3_BASE + filename
    local_extract_path = EXTRACT_DIR / f"bldg{padded}"
    models_dir = local_extract_path / "models"
    zip_file = local_extract_path / filename

    if building_done(local_extract_path):
        return bid, "skipped"

    try:
        download_s3(s3_url, zip_file)
        unzip_to_dir(zip_file, models_dir)
        copy_weather_and_measures(local_extract_path)
        patch_osw(local_extract_path)
        # zip_file.unlink(missing_ok=True)  # optionally delete zip
        return bid, "ok"
    except zipfile.BadZipFile as e:
        return bid, f"bad_zip: {e}"
    except subprocess.CalledProcessError as e:
        return bid, f"download_fail: {e}"
    except Exception as e:
        return bid, f"error: {e}"

# ========================
# MAIN (parallel)
# ========================
def main():
    ensure_base_files()
    print(f"Using downloader: {'boto3 (multipart)' if _use_boto3 else 'awscli module'}")
    results = {"ok": [], "skipped": [], "failed": []}

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(process_building, bid): bid for bid in BUILDING_IDS}
        for fut in as_completed(futures):
            bid, status = fut.result()
            if status == "ok":
                print(f"✔ {bid} done")
                results["ok"].append(bid)
            elif status == "skipped":
                print(f"↩ {bid} skipped (already processed)")
                results["skipped"].append(bid)
            else:
                print(f"✖ {bid} {status}")
                results["failed"].append((bid, status))

    print("\nSUMMARY")
    print("-------")
    print(f"Completed: {results['ok']}")
    print(f"Skipped:   {results['skipped']}")
    print(f"Failed:    {results['failed']}")

if __name__ == "__main__":
    main()


Using downloader: boto3 (multipart)
✔ 379341 done
✔ 253852 done
✔ 140567 done

SUMMARY
-------
Completed: [379341, 253852, 140567]
Skipped:   []
Failed:    []


### RUN MODELS IN PARALLEL

In [10]:
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

#TEMP
building_ids = [140567, 253852, 379341]   # <--- edit

# Windows
#openstudio_path="C:/openstudio-3.9.0/bin/openstudio.exe"

# Mac
openstudio_path="/Applications/OpenStudio-3.9.0/bin/openstudio"

# Quick check
subprocess.run([openstudio_path, "openstudio_version"], check=True, capture_output=True, text=True)

MAX_PARALLEL = 50
extract_dir = Path("../data/resstock_models")

def run_one(bid: int):
    padded = f"{bid:07d}"
    osw = Path(extract_dir) / f"bldg{padded}" / "workflow_resstock.osw"
    if not osw.exists():
        return bid, 1, f"Missing OSW: {osw}"
    r = subprocess.run([openstudio_path, "run", "-w", str(osw)], capture_output=True, text=True)
    return bid, r.returncode, r.stderr.strip()

print(f"Running {len(building_ids)} models (up to {MAX_PARALLEL} at once)...")
errors = []

with ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
    # start all jobs at once (up to MAX_PARALLEL run concurrently)
    futures = [ex.submit(run_one, bid) for bid in building_ids]

    # handle results as each job finishes
    for i, fut in enumerate(as_completed(futures), 1):
        bid, rc, err = fut.result()        # what run_one() returns
        ok = (rc == 0 and err == "")       # success if no error and rc==0
        print(f"[{i}/{len(building_ids)}] bldg {bid}: {'OK' if ok else 'FAIL'}")
        if not ok:
            errors.append((bid, rc, err))

# after all jobs finish, fail the cell if any failed
if errors:
    msg = " | ".join([f"{bid}: rc={rc}, err={err[:120]}" for bid, rc, err in errors])
    raise AssertionError(f"OpenStudio failed for {len(errors)} model(s): {msg}")

print("All done")


Running 3 models (up to 50 at once)...
[1/3] bldg 379341: OK
[2/3] bldg 253852: OK
[3/3] bldg 140567: OK
All done
